<a href="https://colab.research.google.com/github/lohithshankarappa/bisLAB/blob/main/Week4CuckooSearch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
import math

# 1. DEFINE PROBLEM & INITIALIZE DATA
# Generating a synthetic dataset: 1000 samples, 30 features (only 10 are truly informative)
X, y = make_classification(n_samples=1000, n_features=30, n_informative=10, random_state=42)
N_FEATURES = X.shape[1]

# Fitness Evaluation Function
def evaluate_fitness(nest, X, y, w1=0.99, w2=0.01):
    selected_indices = np.where(nest == 1)[0]

    # Penalty if no features are selected
    if len(selected_indices) == 0:
        return 1.0

    # Train classifier using only selected features (3-fold cross-validation)
    clf = RandomForestClassifier(n_estimators=10, random_state=42, n_jobs=-1)
    scores = cross_val_score(clf, X[:, selected_indices], y, cv=3, scoring='accuracy')

    error_rate = 1.0 - np.mean(scores)
    feature_ratio = len(selected_indices) / N_FEATURES

    # Fitness Function (Minimization)
    return (w1 * error_rate) + (w2 * feature_ratio)

# Lévy Flight Step Generation
def levy_flight(Lambda=1.5):
    # Mantegna algorithm to generate step length using standard math library
    sigma_num = (math.gamma(1 + Lambda) * np.sin(np.pi * Lambda / 2))
    sigma_den = (math.gamma((1 + Lambda) / 2) * Lambda * 2**((Lambda - 1) / 2))
    sigma_u = (sigma_num / sigma_den) ** (1 / Lambda)

    u = np.random.normal(0, sigma_u, N_FEATURES)
    v = np.random.normal(0, 1, N_FEATURES)
    step = u / (np.abs(v) ** (1 / Lambda))
    return step


# Sigmoid Transfer Function for Binary Mapping
def sigmoid_transfer(continuous_vector):
    return 1 / (1 + np.exp(-continuous_vector))

# 2. RUN BINARY CUCKOO SEARCH
def binary_cuckoo_search(n_nests=15, p_a=0.25, max_iter=20):
    # 3. Initialize Population (Random Binary Masks) & Continuous States
    nests = np.random.randint(2, size=(n_nests, N_FEATURES))
    nests_continuous = np.random.uniform(-3, 3, size=(n_nests, N_FEATURES))

    # 4. Evaluate Initial Fitness
    fitness = np.array([evaluate_fitness(nest, X, y) for nest in nests])

    best_idx = np.argmin(fitness)
    best_nest = nests[best_idx].copy()
    best_fitness = fitness[best_idx]

    # 7. Iteration Loop
    for iteration in range(max_iter):
        # 5. Generate New Solutions via Lévy Flights
        for i in range(n_nests):
            step = levy_flight()
            # Mutate continuous state using step size alpha=0.1
            nests_continuous[i] += 0.1 * step * (nests_continuous[i] - nests_continuous[best_idx])

            # Map back to binary space via Sigmoid
            s_val = sigmoid_transfer(nests_continuous[i])
            new_nest = np.where(np.random.rand(N_FEATURES) < s_val, 1, 0)

            new_fitness = evaluate_fitness(new_nest, X, y)

            # Greedy replacement selection
            j = np.random.randint(0, n_nests)
            if new_fitness < fitness[j]:
                nests[j] = new_nest
                fitness[j] = new_fitness

        # 6. Abandon Worst Nests (p_a rate)
        worst_indices = np.argsort(fitness)[-int(p_a * n_nests):]
        for idx in worst_indices:
            nests[idx] = np.random.randint(2, size=N_FEATURES)
            nests_continuous[idx] = np.random.uniform(-3, 3, size=N_FEATURES)
            fitness[idx] = evaluate_fitness(nests[idx], X, y)

        # Update Global Best Solution
        current_best_idx = np.argmin(fitness)
        if fitness[current_best_idx] < best_fitness:
            best_fitness = fitness[current_best_idx]
            best_nest = nests[current_best_idx].copy()

        print(f"Iteration {iteration+1}/{max_iter} - Best Fitness: {best_fitness:.4f}")

    return best_nest, best_fitness

# 8. OUTPUT THE BEST SOLUTION
best_features, best_fit = binary_cuckoo_search()
selected_features_list = np.where(best_features == 1)[0]

print("\n--- OPTIMIZATION RESULTS ---")
print(f"Total Features Original Dataset: {N_FEATURES}")
print(f"Selected Features Subsets Index: {selected_features_list}")
print(f"Count of Retained Features     : {len(selected_features_list)}")
print(f"Minimized Multi-objective Cost : {best_fit:.4f}")


Iteration 1/20 - Best Fitness: 0.1670
Iteration 2/20 - Best Fitness: 0.1670
Iteration 3/20 - Best Fitness: 0.1228
Iteration 4/20 - Best Fitness: 0.1228
Iteration 5/20 - Best Fitness: 0.1228
Iteration 6/20 - Best Fitness: 0.1228
Iteration 7/20 - Best Fitness: 0.1228
Iteration 8/20 - Best Fitness: 0.1228
Iteration 9/20 - Best Fitness: 0.1228
Iteration 10/20 - Best Fitness: 0.1228
Iteration 11/20 - Best Fitness: 0.1228
Iteration 12/20 - Best Fitness: 0.1228
Iteration 13/20 - Best Fitness: 0.1228
Iteration 14/20 - Best Fitness: 0.1119
Iteration 15/20 - Best Fitness: 0.1119
Iteration 16/20 - Best Fitness: 0.1119
Iteration 17/20 - Best Fitness: 0.1119
Iteration 18/20 - Best Fitness: 0.1119
Iteration 19/20 - Best Fitness: 0.1119
Iteration 20/20 - Best Fitness: 0.1119

--- OPTIMIZATION RESULTS ---
Total Features Original Dataset: 30
Selected Features Subsets Index: [ 0  4  5  6  8  9 11 14 16 17 19 20 21 22 23 24 25 28]
Count of Retained Features     : 18
Minimized Multi-objective Cost : 0.111